In [2]:
import argparse
import os
import pickle
import time

from importlib import metadata
import torch
try:
    try:
        if metadata.version("rsl-rl"):
            raise ImportError
    except metadata.PackageNotFoundError:
        if metadata.version("rsl-rl-lib") != "3.1.1":  #2.2.4
            raise ImportError
except (metadata.PackageNotFoundError, ImportError) as e:
    raise ImportError("Please uninstall 'rsl_rl' and install 'rsl-rl-lib==2.2.4'.") from e
from rsl_rl.runners import OnPolicyRunner

In [3]:
from bp000_env_cnoid import BP000Env as RLEnv

In [4]:
# 任意設定項目
# exp_name = 'collision-walking-rand'  # ckpt = 4000
# exp_name = 'friction-walking-fractal-kp2000kd50'
# exp_name = 'friction-walking-flat-kp2000kd50'  # ckpt = 20000
# exp_name = 'friction-walking-fractal-kp2000kd50-action_rate-0.01'  # ckpt = 20000
# exp_name = 'friction-walking-fractal-kp2000kd50-linvel-4'
exp_name = 'friction-walking-fractalall-kp2000kd50'
ckpt = 1000

action_scale = 1.0 # 動作のスケールを調整

In [5]:
# 既存のセルを置き換え
import pandas as pd
import numpy as np

# データ収集用のリスト
# action_data = []
obs_data = []
torque_data = []
step_data = []

# CSVファイルの準備
csv_filename = f'obs_data/{exp_name}_step_data.csv'
os.makedirs('obs_data', exist_ok=True)

In [6]:
def _obs_vec(obs):
    # TensorDict or dict → 'policy' を優先
    if isinstance(obs, dict) or hasattr(obs, "get"):
        if "policy" in obs:
            obs = obs["policy"]
    if torch.is_tensor(obs):
        return obs.detach().cpu().numpy().ravel()
    return np.asarray(obs, dtype=np.float32).ravel()

In [7]:
## set robot path fix collisiton 
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # /userdir
robot_path = os.path.join(ROOT, "userdir", "humanoid_research_k", "robots", "kawada_base.simple_collision.urdf")

In [8]:
log_dir = f"logs/{exp_name}"
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f"logs/{exp_name}/cfgs.pkl", "rb"))
reward_cfg["reward_scales"] = {}

In [9]:
## override
# env_cfg["episode_length_s"] = 20.0
# command_cfg["lin_vel_x_range"] = [0.5, 0.5]
# env_cfg['dt'] = 0.001
# env_cfg['substeps'] = 10
# env_cfg["kd"] = 50
env_cfg['base_roll_noise'] = [0,0]
env_cfg['base_pitch_noise'] = [0,0]
env_cfg['termination_if_roll_greater_than'] = 150
env_cfg['termination_if_pitch_greater_than'] = 150
env_cfg['rotorInertia'] = 0.9
env_cfg["base_init_pos"] = [0.0, 0.0, 0.64]

In [10]:
env_cfg



{'num_actions': 12,
 'default_joint_angles': {'R_HIP_Y': 0.0,
  'R_HIP_R': 0.0,
  'R_HIP_P': -0.8,
  'R_KNEE': 1.6,
  'R_ANKLE_P': -0.8,
  'R_ANKLE_R': 0.0,
  'L_HIP_Y': 0.0,
  'L_HIP_R': 0.0,
  'L_HIP_P': -0.8,
  'L_KNEE': 1.6,
  'L_ANKLE_P': -0.8,
  'L_ANKLE_R': 0.0},
 'joint_names': ['R_HIP_Y',
  'R_HIP_R',
  'R_HIP_P',
  'R_KNEE',
  'R_ANKLE_P',
  'R_ANKLE_R',
  'L_HIP_Y',
  'L_HIP_R',
  'L_HIP_P',
  'L_KNEE',
  'L_ANKLE_P',
  'L_ANKLE_R'],
 'kp': 2000.0,
 'kd': 50.0,
 'termination_if_roll_greater_than': 150,
 'termination_if_pitch_greater_than': 150,
 'base_init_pos': [0.0, 0.0, 0.64],
 'base_init_quat': [1.0, 0.0, 0.0, 0.0],
 'episode_length_s': 20.0,
 'resampling_time_s': 4.0,
 'action_scale': 0.25,
 'simulate_action_latency': True,
 'clip_actions': 100.0,
 'dt': 0.01,
 'substeps': 10,
 'rotorInertia': 0.9,
 'base_roll_noise': [0, 0],
 'base_pitch_noise': [0, 0],
 'domain_rand': {'friction': [0.1, 2.0],
  'restitution': [0.0, 0.5],
  'kp': [15000.0, 25000.0],
  'kd': [40.0, 80.0

In [11]:
env = RLEnv(
    num_envs=1,
    env_cfg=env_cfg,
    obs_cfg=obs_cfg,
    reward_cfg=reward_cfg,
    command_cfg=command_cfg,
    dt=env_cfg['dt'],
    substeps=env_cfg['substeps'],
    show_viewer=True,
    robot_urdf_path=robot_path,
)

In [12]:
runner = OnPolicyRunner(env, train_cfg, log_dir, device='cuda')
resume_path = os.path.join(log_dir, f"model_{ckpt}.pt")
runner.load(resume_path)
policy = runner.get_inference_policy(device='cuda')

obs, _ = env.reset()
cnt = 0

torques = env.sim.sbody.getTorques()

print("obs : ", obs["policy"])

# データを記録
step_data.append(cnt)
obs_data.append(_obs_vec(obs))
torque_data.append(torques.copy())

cnt += 1

--------------------------------------------------------------------------------
Resolved observation sets: 
	 policy :  ['policy']
	 critic :  ['policy']
--------------------------------------------------------------------------------
Actor MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=12, bias=True)
)
Critic MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)
obs :  tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        

In [13]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 1
Original actions :  tensor([[ 0.2902,  0.1680,  0.7189,  2.0236, -1.7501, -1.0212, -0.0170,  0.1508,
          0.5418,  1.7538, -1.2858,  0.9067]], device='cuda:0')
Scaled actions :  tensor([[ 0.2902,  0.1680,  0.7189,  2.0236, -1.7501, -1.0212, -0.0170,  0.1508,
          0.5418,  1.7538, -1.2858,  0.9067]], device='cuda:0')


/userdir/irsl_rl/rl_env_base.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.exact_actions = torch.tensor(actions, device=self.device, dtype=torch.float32) ## copy
/userdir/irsl_rl/rl_env_cnoid.py:103: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  self.dof_pos = torch.tensor([self.convAnglesToGenesis(sbody.angleVector())]).to(torch.float32).to(self.device)


obs :  tensor([[ 1.2169e-05, -1.2488e-02,  1.7362e-05,  1.1224e-10, -1.6308e-20,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00, -1.1109e-07,
         -1.5817e-07, -4.7266e-05,  2.2531e-04, -1.2803e-04,  2.3233e-07,
          1.6094e-07, -7.5484e-09, -4.7386e-05,  2.2554e-04, -1.2821e-04,
         -1.3234e-06, -5.5543e-06, -7.9086e-06, -2.3640e-03,  1.1269e-02,
         -6.4021e-03,  1.1617e-05,  8.0469e-06, -3.7742e-07, -2.3695e-03,
          1.1281e-02, -6.4100e-03, -6.6171e-05,  2.9022e-01,  1.6805e-01,
          7.1889e-01,  2.0236e+00, -1.7501e+00, -1.0212e+00, -1.6982e-02,
          1.5081e-01,  5.4182e-01,  1.7538e+00, -1.2858e+00,  9.0668e-01]],
       device='cuda:0')
torques: [-2.97352313e-17  3.47214850e-16 -1.31003155e-05  2.93246369e-05
 -1.25008273e-05  2.51737627e-16 -2.85411999e-17 -9.57523010e-17
 -1.31003155e-05  2.93246369e-05 -1.25008273e-05 -4.99118832e-17]
データ収集: step 2


In [14]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 2
Original actions :  tensor([[ 0.4316, -1.2186,  0.3860,  3.5108, -3.1070, -3.3995,  0.6902,  1.2324,
         -0.0613,  0.2633, -2.9142,  0.7873]], device='cuda:0')
Scaled actions :  tensor([[ 0.4316, -1.2186,  0.3860,  3.5108, -3.1070, -3.3995,  0.6902,  1.2324,
         -0.0613,  0.2633, -2.9142,  0.7873]], device='cuda:0')
obs :  tensor([[-6.7750e-02, -2.9519e-01, -3.4038e-02, -6.7841e-03,  1.6155e-03,
         -9.9998e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  6.7070e-03,
          3.1882e-03,  7.7997e-03,  1.0522e-02, -1.3709e-02, -1.2190e-02,
         -2.8501e-04,  3.0328e-03,  7.7734e-03,  1.0143e-02, -1.3711e-02,
          1.2144e-02,  5.5758e-02,  2.7346e-02,  7.3220e-02,  8.4108e-02,
         -1.1822e-01, -1.1082e-01, -2.4626e-03,  2.5353e-02,  7.2915e-02,
          8.1018e-02, -1.1823e-01,  1.1048e-01,  4.3158e-01, -1.2186e+00,
          3.8596e-01,  3.5108e+00, -3.1070e+00, -3.3995e+00,  6.9015e-01,
          1.2324e+00, -6.1292e-02,  2.6335e-01, -2.9142e+00,  7.8

In [15]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 3
Original actions :  tensor([[-0.1107, -0.9782,  0.6091,  2.8324, -4.7452, -3.2898, -0.1441,  1.0842,
          0.0423,  0.4502, -3.3565,  1.2530]], device='cuda:0')
Scaled actions :  tensor([[-0.1107, -0.9782,  0.6091,  2.8324, -4.7452, -3.2898, -0.1441,  1.0842,
          0.0423,  0.4502, -3.3565,  1.2530]], device='cuda:0')
obs :  tensor([[-7.6274e-02, -3.1022e-01, -2.0735e-01, -1.9037e-02,  4.4580e-03,
         -9.9981e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  2.5070e-02,
          4.5360e-03,  2.3378e-02,  3.4719e-02, -4.9654e-02, -4.6361e-02,
          1.0138e-02,  1.2509e-02,  1.9201e-02,  2.8466e-02, -4.9461e-02,
          4.5494e-02,  1.1380e-01, -9.9092e-03,  7.9811e-02,  1.5125e-01,
         -2.3002e-01, -2.2000e-01,  9.6788e-02,  6.5738e-02,  4.4356e-02,
          9.6948e-02, -2.2827e-01,  2.0417e-01, -1.1072e-01, -9.7823e-01,
          6.0905e-01,  2.8324e+00, -4.7452e+00, -3.2898e+00, -1.4410e-01,
          1.0842e+00,  4.2273e-02,  4.5024e-01, -3.3565e+00,  1.2

In [16]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 4
Original actions :  tensor([[-0.3674, -0.7646,  0.5559,  1.6574, -5.5200, -2.9574, -0.3017,  1.1573,
         -0.0537,  0.1702, -3.1531,  1.3549]], device='cuda:0')
Scaled actions :  tensor([[-0.3674, -0.7646,  0.5559,  1.6574, -5.5200, -2.9574, -0.3017,  1.1573,
         -0.0537,  0.1702, -3.1531,  1.3549]], device='cuda:0')
obs :  tensor([[-9.5547e-02, -3.8538e-01, -1.3791e-01, -3.3321e-02,  7.4341e-03,
         -9.9942e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  3.9019e-02,
         -1.4768e-03,  4.2905e-02,  7.1659e-02, -1.0797e-01, -1.0241e-01,
          2.0714e-02,  2.9458e-02,  2.7731e-02,  5.3172e-02, -1.0683e-01,
          9.7477e-02,  2.8868e-02, -4.4943e-02,  1.0852e-01,  2.1180e-01,
         -3.4194e-01, -3.2951e-01,  2.0777e-02,  9.6411e-02,  4.1433e-02,
          1.4943e-01, -3.3258e-01,  2.9859e-01, -3.6737e-01, -7.6456e-01,
          5.5589e-01,  1.6574e+00, -5.5200e+00, -2.9574e+00, -3.0165e-01,
          1.1573e+00, -5.3721e-02,  1.7020e-01, -3.1531e+00,  1.3

In [17]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 5
Original actions :  tensor([[-0.1459, -0.6224,  0.6926,  0.7140, -6.2645, -2.6342, -0.0712,  0.9553,
         -0.0590,  0.0564, -2.9159,  1.2900]], device='cuda:0')
Scaled actions :  tensor([[-0.1459, -0.6224,  0.6926,  0.7140, -6.2645, -2.6342, -0.0712,  0.9553,
         -0.0590,  0.0564, -2.9159,  1.2900]], device='cuda:0')
obs :  tensor([[-0.1744, -0.3511, -0.0166, -0.0481,  0.0129, -0.9988,  1.0000,  0.0000,
          0.0000,  0.0338, -0.0140,  0.0643,  0.1202, -0.1887, -0.1803,  0.0159,
          0.0513,  0.0352,  0.0835, -0.1840,  0.1626, -0.0712, -0.0779,  0.1050,
          0.2639, -0.4537, -0.4389, -0.0550,  0.1212,  0.0335,  0.1503, -0.4300,
          0.3346, -0.1459, -0.6224,  0.6926,  0.7140, -6.2645, -2.6342, -0.0712,
          0.9553, -0.0590,  0.0564, -2.9159,  1.2900]], device='cuda:0')
torques: [-200.         -200.          -62.62986574   80.17141658 -200.
 -200.         -135.76521007  200.         -131.74766196 -200.
 -200.           30.79400705]
データ収集: step 6


In [18]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 6
Original actions :  tensor([[-0.1331, -0.4253,  0.6806, -0.1865, -6.9521, -2.0513,  0.0163,  0.5666,
          0.1183,  0.0177, -2.5995,  1.1988]], device='cuda:0')
Scaled actions :  tensor([[-0.1331, -0.4253,  0.6806, -0.1865, -6.9521, -2.0513,  0.0163,  0.5666,
          0.1183,  0.0177, -2.5995,  1.1988]], device='cuda:0')
obs :  tensor([[-2.4039e-01, -2.6750e-01,  6.3670e-04, -6.0804e-02,  2.1247e-02,
         -9.9792e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  1.5886e-02,
         -3.3327e-02,  8.8180e-02,  1.6819e-01, -2.9162e-01, -2.7808e-01,
          5.0796e-03,  7.8841e-02,  4.0319e-02,  1.1211e-01, -2.8047e-01,
          2.2853e-01, -9.2911e-02, -1.0934e-01,  1.2434e-01,  2.0831e-01,
         -5.6468e-01, -5.1612e-01, -4.7575e-02,  1.4982e-01,  1.5402e-02,
          1.3271e-01, -5.1185e-01,  3.1011e-01, -1.3307e-01, -4.2527e-01,
          6.8064e-01, -1.8652e-01, -6.9521e+00, -2.0513e+00,  1.6326e-02,
          5.6661e-01,  1.1826e-01,  1.7707e-02, -2.5995e+00,  1.1

In [19]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 7
Original actions :  tensor([[ 0.0196,  0.0495,  0.8144, -0.4224, -7.3653, -1.2333, -0.0209,  0.1085,
          0.2368,  0.2240, -2.2857,  1.0747]], device='cuda:0')
Scaled actions :  tensor([[ 0.0196,  0.0495,  0.8144, -0.4224, -7.3653, -1.2333, -0.0209,  0.1085,
          0.2368,  0.2240, -2.2857,  1.0747]], device='cuda:0')
obs :  tensor([[-2.7196e-01, -1.8886e-01,  3.2423e-02, -6.9934e-02,  3.1739e-02,
         -9.9705e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -2.9031e-03,
         -5.6323e-02,  1.1532e-01,  2.0017e-01, -4.1673e-01, -3.7983e-01,
         -1.6242e-03,  1.0773e-01,  4.4212e-02,  1.3510e-01, -3.8745e-01,
          2.8435e-01, -9.0045e-02, -1.1749e-01,  1.3918e-01,  1.2136e-01,
         -6.7523e-01, -4.9342e-01, -2.2273e-02,  1.3503e-01,  2.0062e-02,
          9.9487e-02, -5.3991e-01,  2.2694e-01,  1.9555e-02,  4.9475e-02,
          8.1442e-01, -4.2239e-01, -7.3653e+00, -1.2333e+00, -2.0900e-02,
          1.0846e-01,  2.3678e-01,  2.2399e-01, -2.2857e+00,  1.0

In [20]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 8
Original actions :  tensor([[ 0.2396,  0.1851,  0.9049, -0.4586, -7.5108, -0.6348, -0.1050, -0.2989,
          0.3944,  0.2653, -1.9886,  0.7295]], device='cuda:0')
Scaled actions :  tensor([[ 0.2396,  0.1851,  0.9049, -0.4586, -7.5108, -0.6348, -0.1050, -0.2989,
          0.3944,  0.2653, -1.9886,  0.7295]], device='cuda:0')
obs :  tensor([[-3.0748e-01, -1.1676e-01,  6.7181e-02, -7.6010e-02,  4.3558e-02,
         -9.9616e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -1.4970e-02,
         -7.4216e-02,  1.4501e-01,  2.1519e-01, -5.6393e-01, -4.6648e-01,
         -4.8814e-03,  1.2920e-01,  5.0308e-02,  1.5108e-01, -4.8288e-01,
          3.1676e-01, -3.8164e-02, -6.5992e-02,  1.5159e-01,  3.8005e-02,
         -7.8579e-01, -3.8399e-01, -1.2229e-02,  8.5217e-02,  3.7492e-02,
          6.6046e-02, -4.2328e-01,  1.0913e-01,  2.3962e-01,  1.8512e-01,
          9.0490e-01, -4.5861e-01, -7.5108e+00, -6.3475e-01, -1.0502e-01,
         -2.9895e-01,  3.9441e-01,  2.6531e-01, -1.9886e+00,  7.2

In [21]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 9
Original actions :  tensor([[ 0.3232,  0.2784,  0.8871, -0.0892, -7.5304, -0.2220, -0.3004, -0.3664,
          0.5687,  0.2807, -1.7389,  0.4612]], device='cuda:0')
Scaled actions :  tensor([[ 0.3232,  0.2784,  0.8871, -0.0892, -7.5304, -0.2220, -0.3004, -0.3664,
          0.5687,  0.2807, -1.7389,  0.4612]], device='cuda:0')
obs :  tensor([[-2.9220e-01, -6.5840e-02,  3.8350e-02, -7.9645e-02,  5.5833e-02,
         -9.9526e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -1.3385e-02,
         -8.1853e-02,  1.7781e-01,  2.1365e-01, -7.3326e-01, -5.3123e-01,
         -9.6102e-03,  1.4289e-01,  6.1590e-02,  1.6147e-01, -5.5683e-01,
          3.2656e-01,  4.1156e-02, -1.6523e-02,  1.6554e-01, -4.3612e-02,
         -8.9629e-01, -2.7456e-01, -3.3460e-02,  6.1706e-02,  6.1398e-02,
          4.1629e-02, -3.2039e-01,  2.4343e-03,  3.2321e-01,  2.7845e-01,
          8.8715e-01, -8.9207e-02, -7.5304e+00, -2.2203e-01, -3.0041e-01,
         -3.6639e-01,  5.6872e-01,  2.8067e-01, -1.7389e+00,  4.6

In [22]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 10
Original actions :  tensor([[ 0.2105,  0.2888,  0.9216, -0.0654, -7.6130,  0.0131, -0.3861, -0.3501,
          0.6250,  0.4309, -1.6222,  0.3933]], device='cuda:0')
Scaled actions :  tensor([[ 0.2105,  0.2888,  0.9216, -0.0654, -7.6130,  0.0131, -0.3861, -0.3501,
          0.6250,  0.4309, -1.6222,  0.3933]], device='cuda:0')
obs :  tensor([[-2.6778e-01, -5.5928e-02,  5.2504e-02, -8.2295e-02,  6.7028e-02,
         -9.9435e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  2.2224e-03,
         -7.9584e-02,  2.1208e-01,  1.9645e-01, -9.2465e-01, -5.7412e-01,
         -2.2920e-02,  1.5397e-01,  7.6134e-02,  1.6726e-01, -6.0968e-01,
          3.1583e-01,  9.9022e-02,  3.4184e-02,  1.6482e-01, -1.1887e-01,
         -1.0065e+00, -1.6528e-01, -8.6397e-02,  5.0981e-02,  7.3765e-02,
          1.9381e-02, -2.1929e-01, -9.9496e-02,  2.1048e-01,  2.8878e-01,
          9.2164e-01, -6.5436e-02, -7.6130e+00,  1.3057e-02, -3.8613e-01,
         -3.5014e-01,  6.2503e-01,  4.3091e-01, -1.6222e+00,  3.

In [23]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 11
Original actions :  tensor([[-0.0288,  0.2449,  1.1411, -0.3562, -7.8294,  0.2385, -0.2890, -0.4190,
          0.4381,  0.7974, -2.2211,  0.4396]], device='cuda:0')
Scaled actions :  tensor([[-0.0288,  0.2449,  1.1411, -0.3562, -7.8294,  0.2385, -0.2890, -0.4190,
          0.4381,  0.7974, -2.2211,  0.4396]], device='cuda:0')
obs :  tensor([[-2.5507e-01, -4.4997e-02,  1.1901e-01, -8.4276e-02,  7.7642e-02,
         -9.9341e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  2.3151e-02,
         -6.7324e-02,  2.4454e-01,  1.6487e-01, -1.1348e+00, -5.9517e-01,
         -4.5365e-02,  1.6169e-01,  9.2790e-02,  1.6940e-01, -6.4226e-01,
          2.8447e-01,  1.0313e-01,  8.3110e-02,  1.5236e-01, -1.8915e-01,
         -1.0965e+00, -5.6194e-02, -1.3241e-01,  2.8618e-02,  8.9010e-02,
          3.1488e-03, -1.1672e-01, -2.0389e-01, -2.8766e-02,  2.4492e-01,
          1.1411e+00, -3.5619e-01, -7.8294e+00,  2.3851e-01, -2.8897e-01,
         -4.1895e-01,  4.3806e-01,  7.9735e-01, -2.2211e+00,  4.

In [45]:
# 既存のforループを置き換え
num_steps = 10
for i in range(num_steps):
    with torch.no_grad():
        actions = policy(obs)
        
        # アクションスケーリング
        scaled_actions = actions * action_scale
        obs, rews, dones, infos = env.step(scaled_actions)  # スケール済みを使用
        torques = env.sim.sbody.getTorques()
        
        # データを記録
        step_data.append(cnt)
        obs_data.append(_obs_vec(obs))
        torque_data.append(torques.copy())
        
        # デバッグ表示（最初の数ステップのみ）
        if i < 3:
            print(f"Step {i}: Original action max={actions.max():.3f}, "
                  f"Scaled action max={scaled_actions.max():.3f}")
        
        if i % 20 == 0:
            print(f"Step {i+1}/{num_steps}, Total steps: {cnt}")
            print("steps:",cnt)
            print("actions :",scaled_actions)
            print("target_dof_pos:",env.target_dof_pos)
        
        cnt += 1

print(f"データ収集完了: {num_steps} steps collected with action_scale={action_scale}")

Step 0: Original action max=1.397, Scaled action max=1.397
Step 1/10, Total steps: 202
steps: 202
actions : tensor([[ 1.1352,  0.1784, -0.1447,  0.5888, -6.0607, -1.6868,  0.2530, -0.2939,
         -0.5836,  1.3966, -1.6205, -1.0531]], device='cuda:0')
target_dof_pos: tensor([[ 0.2497, -0.0919, -0.9927,  1.7947, -2.3589, -0.3939,  0.1109, -0.2257,
         -0.7948,  2.0490, -0.7875, -0.4012]], device='cuda:0')
Step 1: Original action max=1.866, Scaled action max=1.866
Step 2: Original action max=2.361, Scaled action max=2.361
データ収集完了: 10 steps collected with action_scale=1.0


In [25]:
# for i in range(500):
#     with torch.no_grad():
#         actions = policy(obs)
#         scaled_actions = actions * action_scale
#         obs, rews, dones, infos = env.step(scaled_actions)

In [26]:
env.sim.stop()

In [44]:
# 最もシンプルな保存方法
def save_simple_csv():
    if not step_data:
        print("データがありません")
        return
    
    # 基本的な辞書形式でデータを整理
    data_dict = {'step': step_data}
    
    # # Actionデータ
    # action_array = np.array(action_data)
    # for i in range(action_array.shape[1]):
    #     data_dict[f'action_{i}'] = action_array[:, i]
    
    # Observationデータ
    obs_array = np.array(obs_data)
    for i in range(obs_array.shape[1]):
        data_dict[f'obs_{i}'] = obs_array[:, i]
    
    # Torqueデータ
    torque_array = np.array(torque_data)
    for i in range(torque_array.shape[1]):
        data_dict[f'torque_{i}'] = torque_array[:, i]
    
    # DataFrameを作成して保存
    df = pd.DataFrame(data_dict)
    csv_filename = f'obs_data/cnoid_{exp_name}_ckpt{ckpt}_scale{action_scale}_rotorInertia0.9.csv'
    df.to_csv(csv_filename, index=False)
    
    print(f"シンプル版を保存: {csv_filename}")
    print(f"データ形状: {df.shape}")
    
    return df

# シンプル版を実行
df_simple = save_simple_csv()

シンプル版を保存: obs_data/cnoid_friction-walking-fractal-kp2000kd50_ckpt100_scale1.0_rotorInertia0.9.csv
データ形状: (192, 58)
